# Module 2 · Lesson 01: Zero-Shot Prompting

**Zero-shot prompting** means asking the model to perform a task *without any examples*.
The model relies entirely on its training data and your instructions.

## What you will learn
1. Effective zero-shot prompt patterns
2. Output **format specification**
3. **Role/persona** assignment
4. Zero-shot **classification**
5. **Structured output** (JSON) extraction
6. Using **constraints** to control output

In [5]:
# ── Setup ──────────────────────────────────────────────
import os
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
 
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
 
if client:
    print("Client is ready")
 

Client is ready


In [6]:
def ask(prompt, system=None, temperature = 0.7, max_tokens=1200):
    msgs = []
    if system:
        msgs.append({"role":"system", "content":system})
    msgs.append({"role":"user", "content":prompt})
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=msgs,
        temperature=temperature,
        max_tokens=max_tokens
    )
    return r.choices[0].message.content

---
## 1. Simple Zero-Shot

The simplest form — just ask directly:

In [7]:
result = ask("What is the capital of France?")
display(Markdown(f"**Q**: What is the capital of France?\n\n**A**: {result}"))

**Q**: What is the capital of France?

**A**: The capital of France is Paris.

---
## 2. Format Specification

Tell the model **exactly** what format you want:

In [10]:
prompt = """Extract the email address from this text:
 
Hi, you can reach me at john.doe@example.com for more information.
"""
result = ask(prompt, max_tokens=50)
display(Markdown(f"**Extracted:** `{result.strip()}`"))

**Extracted:** `The email address extracted from the text is: john.doe@example.com`

---
## 3. Role / Persona Assignment

The system prompt sets *who* the model should be:

In [11]:
result = ask(
  prompt="Explain what an API is to a non-technical person.",
  system="You are a professional technical writer who explains complex concepts simply."
)

display(Markdown(f"### Technical Writer:\n\n{result}"))

### Technical Writer:

Sure! Think of an API (which stands for Application Programming Interface) like a waiter in a restaurant. 

When you go to a restaurant, you look at the menu and decide what you want to order. You tell the waiter your choice, and then the waiter goes to the kitchen to communicate your order to the chef. Once your food is ready, the waiter brings it back to you.

In this analogy:

- **The restaurant** is like a software system or service.
- **The menu** is like the API documentation, which tells you what options are available and how to request them.
- **The waiter** is the API itself, acting as a middleman that takes your requests and translates them into a format the system can understand.
- **The kitchen** is where the actual processing happens, similar to how the backend of a software application works.
- **Your food** is the information or service you requested.

So, an API allows different software systems to communicate with each other easily, just like a waiter helps you get your food from the kitchen without you needing to know how to cook it yourself. It simplifies the process of making requests and receiving responses between different applications!

---
## 4. Zero-Shot Classification

LLMs are excellent **zero-shot classifiers**. No training data needed!

In [13]:
texts = [
  "I absoluteley love this product! It has changed my life for the better.",
  "The shipping was delayed and the item arrived damaged. Very disappointed.",
  "It's okay, nothing special but it does the job. I might buy it again if it's on sale."
]

print(f"{'Text':<55} {'Sentiment'}")
print("-" * 101)

for text in texts:
  prompt = f"Classify the sentiment as exactly one world: Positive, Negative, or Neutral.\n\nText: {text}\n\n:"
  sentiment = ask(prompt, temperature=0, max_tokens=10).strip().lower()
  emoji = {"positive": "🟢", "negative": "🔴", "neutral": "🟡"}.get(sentiment, "⚪")
  print(f"{text[:52]+'...':<55} {emoji} {sentiment}")

Text                                                    Sentiment
-----------------------------------------------------------------------------------------------------
I absoluteley love this product! It has changed my l... 🟢 positive
The shipping was delayed and the item arrived damage... 🔴 negative
It's okay, nothing special but it does the job. I mi... 🟡 neutral


# Role - Context - Structure

In [14]:
result = ask(
  prompt="Explain what an API is to an 8-year-old child. Do it in 3 short sentences.",
  system="You are my school teacher who loves analogies and explain everything in easy way."
)

display(Markdown(f"### Technical Writer:\n\n{result}"))

### Technical Writer:

Imagine you have a toy store, and you want to order a new toy. An API is like a special order form that you give to the store so they know exactly what toy you want and how many. When they get your order, they send you back the toy you asked for, just like magic!

> 💡 Use `temperature=0` for classification to get deterministic, consistent results.

---
## 5. Structured Output (JSON)

In [18]:
import json
 
prompt = """Extract information from this text and return as valid JSON:
 
Text: "Meeting with Sarah Johnson scheduled for March 15, 2026 at 2:30PM to discuss te Q1 budget report."
 
Return JSON fields: attendee, date, time, topic
 
Do NOT wrap the output in markdown code fences or any other formating.
Return ONLY the raw JSON object, nothing else.
 
JSON:"""
 
result = ask(prompt, temperature=0)
try:
    parsed = json.loads(result)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {result}")
    print("Not valid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [19]:
response_json = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role":"user", "content":prompt}],
    response_format={"type": "json_object"}, # force valid JSON output
    temperature=0
)

new_res = response_json.choices[0].message.content

try:
    parsed = json.loads(new_res)
    display(Markdown(f"```json\n{json.dumps(parsed, indent=2)}\n```"))
except json.JSONDecodeError:
    print(f"Raw output: {new_res}")
    print("Not valid JSON")

```json
{
  "attendee": "Sarah Johnson",
  "date": "2026-03-15",
  "time": "14:30",
  "topic": "Q1 budget report"
}
```

In [20]:
import json
 
response_json = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{"role":"user", "content":"Return a short greeting and a lucky number"}],
    response_format={
        "type":"json_schema",
        "json_schema": {
            "name": "greeting_response",
            "schema": {
                "type": "object",
                "properties": {
                    "greeting": {"type": "string"},
                    "luckyNumber": {"type": "integer"}
                },
                "required": ["greeting", "luckyNumber"],
                "additionalProperties":False
            },
            "strict":True
        }
    },
    temperature=0
)
 
new_res = response_json.choices[0].message.content
parsed = json.loads(new_res)
 
print(json.dumps(parsed, indent=2))

{
  "greeting": "Hello! Wishing you a fantastic day ahead!",
  "luckyNumber": 7
}


In [21]:
import json
 
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Describe an API endpoint for user login."}
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "api_endpoint",
            "schema": {
                "type": "object",
                "properties": {
                    "endpoint": {"type": "string"},
                    "method": {
                        "type": "string",
                        "enum": ["GET", "POST", "PUT", "DELETE"]
                    },
                    "request_body": {
                        "type": "object",
                        "properties": {
                            "email": {"type": "string"},
                            "password": {"type": "string"}
                        },
                        "required": ["email", "password"],
                        "additionalProperties": False
                    },
                    "response": {
                        "type": "object",
                        "properties": {
                            "token": {"type": "string"},
                            "expires_in": {"type": "integer"}
                        },
                        "required": ["token", "expires_in"],
                        "additionalProperties": False
                    }
                },
                "required": ["endpoint", "method", "request_body", "response"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    temperature=0
)
 
parsed = json.loads(response.choices[0].message.content)
print(json.dumps(parsed, indent=2))

{
  "endpoint": "/api/login",
  "method": "POST",
  "request_body": {
    "email": "user@example.com",
    "password": "yourpassword"
  },
  "response": {
    "token": "abc123xyz",
    "expires_in": 3600
  }
}


---
## 6. Constraints

Adding explicit **constraints** controls length, format, and content:

In [26]:
prompt = """Write a product description for a wireless mouse.
 
Constraints:
- Maximum 50 words
- Include at least one benefit
- Do not mention price
- End with a call to action
 
Description:"""
 
result = ask(prompt)
word_count = len(result.split())
 
display(Markdown(f"> {result}"))
print(f"word_count: {word_count}")

> Experience seamless navigation with our sleek wireless mouse, designed for ultimate comfort and precision. Its ergonomic shape reduces hand fatigue, while the advanced sensor ensures smooth tracking on any surface. Elevate your productivity today—upgrade your workspace with this essential tool!

word_count: 40


### Multi-dimentional constraints (style - structure - semantics)

In [28]:
prompt = """
Write a product description for wireless mouse.
 
Constaints:
- 40-60 words
- Exactly 2 sentences
- First sentence: describe features
- Second sentece: emphasize a user benefit
- Include exactly 1 emoji
- Must contain the word "precision"
- Do NOT use passive voice
- End with a call to action
 
Return ONLY the description.
"""
 
result = ask(prompt)
display(Markdown(f" < {result}"))
print(f"Word count: {len(result. split())}")

 < Experience seamless navigation with our wireless mouse, featuring advanced optical sensors for precision tracking, customizable buttons, and an ergonomic design. Elevate your productivity and comfort—grab yours today! 🖱️

Word count: 28


---
## 7. Prompt Gallery: Real-World System Prompts

Let's study system prompts from **real community projects**. Each uses a different technique
to get reliable, high-quality outputs.

| Source | Key Technique |
|--------|---------------|
| Gmail Summarizer | Structured rules + labels |
| Budget Travel Agent | Role + constraint + format |
| Biomedical Summariser | Audience awareness |
| Code Explainer | Section structure |

In [29]:
# ── Prompt Gallery: 4 real-world system prompts ─────────
 
gallery = {
    "Gmail Summarizer": {
        "prompt": """You summarize email threads. For each email:
- Subject line (max 10 words)
- Label: ACTION_REQUIRED | FYI | PROMO | URGENT
- Summary (max 2 sentences)
- Has link: yes/no
Return as a numbered list.""",
        "technique": "Structured rules with labels and constraints",
    },
    "Budget Travel Agent": {
        "prompt": """You are a budget travel advisor. For any destination:
1. List top 5 FREE attractions
2. Suggest 3 budget restaurants (under $15/meal)
3. Give one money-saving local tip
Respond in markdown with headers.""",
        "technique": "Role + numbered constraints + format (markdown)",
    },
    "Biomedical Summariser": {
        "prompt": """Summarize biomedical research articles for a mixed audience:
students, early researchers, and professionals.
- Use bullet points for key findings
- Highlight methodology and sample size
- Note limitations and future directions
Tone: professional, clear, accessible.""",
        "technique": "Audience awareness + structure + tone",
    },
    "Code Explainer": {
        "prompt": """You explain code to developers. Structure your response as:
1) Direct Answer (1-2 sentences)
2) Explanation (why it works)
3) Example (working code snippet)
4) Common Pitfalls (what to avoid)
5) Next Steps (what to learn next)""",
        "technique": "Section-structured output format",
    }
}
 
# Display each prompt with analysis
for name, info in gallery.items():
    print(f"\n{'=' * 60}")
    print(f"  {name}")
    print(f"  Technique: {info['technique']}")
    print(f"{'=' * 60}")
    print(info['prompt'])


  Gmail Summarizer
  Technique: Structured rules with labels and constraints
You summarize email threads. For each email:
- Subject line (max 10 words)
- Label: ACTION_REQUIRED | FYI | PROMO | URGENT
- Summary (max 2 sentences)
- Has link: yes/no
Return as a numbered list.

  Budget Travel Agent
  Technique: Role + numbered constraints + format (markdown)
You are a budget travel advisor. For any destination:
1. List top 5 FREE attractions
2. Suggest 3 budget restaurants (under $15/meal)
3. Give one money-saving local tip
Respond in markdown with headers.

  Biomedical Summariser
  Technique: Audience awareness + structure + tone
Summarize biomedical research articles for a mixed audience:
students, early researchers, and professionals.
- Use bullet points for key findings
- Highlight methodology and sample size
- Note limitations and future directions
Tone: professional, clear, accessible.

  Code Explainer
  Technique: Section-structured output format
You explain code to developers. 

In [30]:
travel_result = ask(
    prompt="I'm visiting Lisbon, Portugal for 3 days on a tight budget.",
    system=gallery["Budget Travel Agent"]["prompt"]
)
 
display(Markdown(f"### Budget Travel Agent Response\n\n{travel_result}"))

### Budget Travel Agent Response

# Budget Travel Guide to Lisbon, Portugal

## Top 5 FREE Attractions
1. **Alfama District**: Wander through the narrow streets of one of Lisbon's oldest neighborhoods, filled with charming houses, local shops, and beautiful viewpoints.
2. **Miradouro da Senhora do Monte**: Enjoy breathtaking panoramic views of the city and the Tagus River from this popular viewpoint, perfect for photography and relaxation.
3. **Praça do Comércio**: Stroll around this historic square by the river, take in the stunning architecture, and enjoy the lively atmosphere.
4. **Belém Cultural Center (CCB)**: Explore contemporary art exhibitions and cultural events at this venue, with free admission to some areas.
5. **Gulbenkian Garden**: Relax in the beautiful gardens of the Calouste Gulbenkian Foundation, which are open to the public and perfect for a picnic.

## Budget Restaurants (Under $15/Meal)
1. **Time Out Market**: While some stalls can be pricey, many offer affordable options. Look for local specialties and grab a quick bite without breaking the bank.
2. **A Cevicheria**: Known for its delicious ceviche, this casual eatery offers reasonably priced dishes that can easily fit within your budget.
3. **Casa da India**: A favorite among locals, this restaurant serves up generous portions of traditional Portuguese dishes, with many options under $15.

## Money-Saving Local Tip
**Use Public Transport**: Lisbon has an efficient public transport system, including trams, buses, and the metro. Consider getting a 24-hour public transport pass, which allows unlimited travel for a single day and can save you money compared to individual tickets. Additionally, walking is a great way to explore the city, especially in areas like Alfama and Bairro Alto.

In [31]:
gmail_result = ask(
    """Here's an email thread:
From: marketing@company.com
Subject: Re: Q2 Campaign Launch — Asset Review Needed
Body: Hi team, please review the attached creatives for the Q2 social campaign.
We need approvals by Friday. The campaign landing page is at https://company.com/q2launch.
Let me know if any changes are needed. Thanks!""",
    system=gallery["Gmail Summarizer"]["prompt"]
)
display(Markdown(f"### Gmail Summarizer Response\n\n{gmail_result}"))
print("\nNotice the consistent labeling and structured bullet format!")

### Gmail Summarizer Response

1. Subject line: Q2 Campaign Launch — Asset Review Needed  
   Label: ACTION_REQUIRED  
   Summary: The team is asked to review the attached creatives for the Q2 social campaign and provide approvals by Friday. The campaign landing page is linked for reference.  
   Has link: yes


Notice the consistent labeling and structured bullet format!


In [32]:
bio_result = ask(
    """A 2024 study published in The Lancet examined the efficacy of a novel mRNA-based
therapeutic vaccine for stage III melanoma. The randomized, double-blind trial enrolled
340 patients across 22 clinical sites. Results showed a 44% reduction in recurrence risk
(HR 0.56, 95% CI 0.40–0.78, p=0.0007) over a 24-month follow-up. Common adverse effects
included fatigue (32%) and injection-site reactions (28%). The authors noted the relatively
short follow-up period and homogeneous population (predominantly Caucasian, median age 58)
as key limitations, and called for larger Phase III trials with diverse cohorts.""",
    system=gallery["Biomedical Summariser"]["prompt"]
)
display(Markdown(f"### Biomedical Summariser Response\n\n{bio_result}"))
print("\nNotice how it highlights methodology, limitations, and stays accessible!")

### Biomedical Summariser Response

**Study Summary: Efficacy of a Novel mRNA-Based Therapeutic Vaccine for Stage III Melanoma**

- **Objective:** To evaluate the efficacy of a new mRNA-based therapeutic vaccine in patients with stage III melanoma.

- **Methodology:**
  - Type of Study: Randomized, double-blind trial.
  - Sample Size: 340 patients.
  - Locations: Conducted across 22 clinical sites.
  - Follow-Up Duration: 24 months.

- **Key Findings:**
  - The mRNA vaccine demonstrated a **44% reduction in recurrence risk** (Hazard Ratio (HR) 0.56, 95% Confidence Interval (CI) 0.40–0.78, p=0.0007).
  - Adverse effects noted included:
    - Fatigue: **32%** of participants reported this side effect.
    - Injection-site reactions: **28%** of participants experienced this.

- **Limitations:**
  - The follow-up period was relatively short (24 months), which may not capture long-term efficacy and safety.
  - The study sample was homogeneous, predominantly consisting of Caucasian participants with a median age of 58, limiting generalizability to more diverse populations.

- **Future Directions:**
  - The authors recommend conducting larger Phase III trials that include more diverse cohorts to better understand the vaccine's efficacy and safety across different demographics.


Notice how it highlights methodology, limitations, and stays accessible!


In [33]:
code_result = ask(
    """Explain this Python code:
result = {k: v for k, v in sorted(data.items(), key=lambda item: item[1], reverse=True)[:5]}""",
    system=gallery["Code Explainer"]["prompt"]
)
display(Markdown(f"### Code Explainer Response\n\n{code_result}"))
print("\nNotice the 5-section structure: Answer → Explanation → Example → Pitfalls → Next Steps!")

### Code Explainer Response

1) **Direct Answer**: This code creates a dictionary called `result` that contains the top 5 key-value pairs from the `data` dictionary, sorted by their values in descending order.

2) **Explanation**: The code uses a dictionary comprehension to iterate over the sorted items of the `data` dictionary. The `sorted()` function sorts the items based on the values (using `item[1]`) in reverse order. By slicing `[:5]`, it selects only the top 5 items after sorting.

3) **Example**:
   ```python
   data = {'a': 10, 'b': 50, 'c': 30, 'd': 40, 'e': 20}
   result = {k: v for k, v in sorted(data.items(), key=lambda item: item[1], reverse=True)[:5]}
   print(result)  # Output: {'b': 50, 'd': 40, 'c': 30, 'e': 20, 'a': 10}
   ```

4) **Common Pitfalls**: A common mistake is assuming that the original dictionary is ordered. If `data` has fewer than 5 items, the code will still work, but `result` will contain fewer items than expected, which could lead to confusion.

5) **Next Steps**: To deepen your understanding, explore Python's data structures such as `collections.Counter` for counting occurrences, and learn about handling ties when sorting data (e.g., using secondary keys).


Notice the 5-section structure: Answer → Explanation → Example → Pitfalls → Next Steps!


> **Exercise:** Pick a business domain you're interested in (e.g., fitness coaching,
> legal review, recipe generation) and write your own system prompt following the
> patterns above. Test it with 3 different user queries.

---
## Key Takeaways 📝

| Technique | When to Use |
|-----------|------------|
| **Direct question** | Simple factual queries |
| **Format specification** | When you need specific output format |
| **Role assignment** | To control tone, expertise, style |
| **Classification** | Categorizing text (use temp=0) |
| **JSON extraction** | Structured data from unstructured text |
| **Constraints** | Controlling length, style, content boundaries |
| **Prompt gallery** | Study real prompts to develop critical analysis skills |

### Zero-Shot Tips
1. Be **specific** about what you want
2. Specify the **output format** explicitly
3. Use **roles/personas** to guide behaviour
4. Add **negative constraints** (what NOT to do)
5. Use `temperature=0` for deterministic tasks

---
**Next:** `02_few_shot_examples.ipynb` — Improve results by providing examples